# 02 — Data Dictionary and Join Plan

## Objective
Define the analytical data model for NOVAres SecureHealth:

- main entities
- primary and foreign keys
- join relationships
- variables by analytical module
- target analytical tables to be built

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print("PROJECT_ROOT:", PROJECT_ROOT)

PROJECT_ROOT: C:\Users\dolivares\Desktop\novares-securehealth


In [2]:
import pandas as pd

from src.data.load_data import (
    load_insured_members,
    load_policies,
    load_providers,
    load_claims,
    load_member_year_features,
    load_provider_month_features,
    load_prospect_survey,
)

In [3]:
members = load_insured_members()
policies = load_policies()
providers = load_providers()
claims = load_claims()
member_year = load_member_year_features()
provider_month = load_provider_month_features()
prospects = load_prospect_survey()

In [4]:
# Entity inventory
entities = pd.DataFrame([
    {"entity": "insured_members", "grain": "1 row per member", "main_key": "member_id", "secondary_key": "policy_id"},
    {"entity": "policies", "grain": "1 row per policy", "main_key": "policy_id", "secondary_key": "member_id"},
    {"entity": "providers", "grain": "1 row per provider", "main_key": "provider_id", "secondary_key": None},
    {"entity": "claims", "grain": "1 row per claim", "main_key": "claim_id", "secondary_key": "member_id / provider_id / policy_id"},
    {"entity": "member_year_features", "grain": "1 row per member-year", "main_key": "member_id + year", "secondary_key": "policy_id"},
    {"entity": "provider_month_features", "grain": "1 row per provider-month", "main_key": "provider_id + month", "secondary_key": None},
    {"entity": "prospect_survey", "grain": "1 row per prospect", "main_key": "prospect_id", "secondary_key": None},
])

entities

,entity,grain,main_key,secondary_key
0,insured_members,1 row per member,member_id,policy_id
1,policies,1 row per policy,policy_id,member_id
2,providers,1 row per provider,provider_id,None
3,claims,1 row per claim,claim_id,member_id / provider_id / policy_id
4,member_year_features,1 row per member-year,member_id + year,policy_id
5,provider_month_features,1 row per provider-month,provider_id + month,None
6,prospect_survey,1 row per prospect,prospect_id,None


In [5]:
# Column inventory by dataset
column_inventory = {
    "insured_members": members.columns.tolist(),
    "policies": policies.columns.tolist(),
    "providers": providers.columns.tolist(),
    "claims": claims.columns.tolist(),
    "member_year_features": member_year.columns.tolist(),
    "provider_month_features": provider_month.columns.tolist(),
    "prospect_survey": prospects.columns.tolist(),
}

for name, cols in column_inventory.items():
    print("=" * 100)
    print(name.upper())
    print("=" * 100)
    for c in cols:
        print("-", c)
    print()

INSURED_MEMBERS
- member_id
- policy_id
- join_date
- age
- age_band
- sex
- region
- city_tier
- socioeconomic_band
- employment_status
- marital_status
- dependents_n
- family_complexity
- smoker_flag
- alcohol_risk_flag
- physical_activity_level
- bmi_group
- chronic_condition_flag
- chronic_condition_count
- chronic_group
- recurrent_medication_flag
- prior_hospitalization_24m_flag
- self_management_adherence
- archetype
- baseline_risk_score
- utilization_propensity
- acute_event_propensity
- misuse_exposure_propensity
- price_sensitivity
- coverage_preference
- network_preference
- preferred_plan_type

POLICIES
- policy_id
- member_id
- policy_start_date
- policy_end_date
- active_flag
- plan_type
- plan_tier
- coverage_scope
- provider_network_type
- deductible_amount
- copay_level
- annual_coverage_limit
- maternity_coverage_flag
- pharmacy_coverage_flag
- chronic_care_program_flag
- premium_monthly
- premium_annual
- underwriting_load_factor
- discount_factor
- recommended_pla

In [6]:
# Proposed joins
join_plan = pd.DataFrame([
    {
        "left_table": "insured_members",
        "right_table": "policies",
        "join_keys": "member_id + policy_id",
        "join_type": "left",
        "expected_cardinality": "1:1",
        "purpose": "Build member master table"
    },
    {
        "left_table": "claims",
        "right_table": "insured_members",
        "join_keys": "member_id",
        "join_type": "left",
        "expected_cardinality": "many:1",
        "purpose": "Enrich claims with member profile"
    },
    {
        "left_table": "claims",
        "right_table": "policies",
        "join_keys": "policy_id",
        "join_type": "left",
        "expected_cardinality": "many:1",
        "purpose": "Enrich claims with plan and premium data"
    },
    {
        "left_table": "claims",
        "right_table": "providers",
        "join_keys": "provider_id",
        "join_type": "left",
        "expected_cardinality": "many:1",
        "purpose": "Enrich claims with provider risk and cost signals"
    },
    {
        "left_table": "member_year_features",
        "right_table": "insured_members",
        "join_keys": "member_id",
        "join_type": "left",
        "expected_cardinality": "many:1",
        "purpose": "Build annual member modeling base"
    },
    {
        "left_table": "provider_month_features",
        "right_table": "providers",
        "join_keys": "provider_id",
        "join_type": "left",
        "expected_cardinality": "many:1",
        "purpose": "Build provider anomaly base"
    },
])

join_plan

,left_table,right_table,join_keys,join_type,expected_cardinality,purpose
0,insured_members,policies,member_id + policy_id,left,1:1,Build member master table
1,claims,insured_members,member_id,left,many:1,Enrich claims with member profile
2,claims,policies,policy_id,left,many:1,Enrich claims with plan and premium data
3,claims,providers,provider_id,left,many:1,Enrich claims with provider risk and cost signals
4,member_year_features,insured_members,member_id,left,many:1,Build annual member modeling base
5,provider_month_features,providers,provider_id,left,many:1,Build provider anomaly base


In [7]:
## Analytical modules and key variables
module_map = pd.DataFrame([
    {
        "module": "Risk Scoring",
        "main_tables": "insured_members, policies, claims, member_year_features",
        "unit_of_analysis": "member_id / member_id-year",
        "key_variables": "age, sex, chronic_condition_flag, chronic_condition_count, baseline_risk_score, utilization_propensity, acute_event_propensity, premium_monthly, deductible_amount, annual_cost"
    },
    {
        "module": "Fraud / Abuse",
        "main_tables": "claims, providers, provider_month_features, member_year_features",
        "unit_of_analysis": "claim_id / member_id / provider_id-month",
        "key_variables": "misuse_exposure_propensity, provider_archetype, fraud_exposure_score, diagnostic_intensity, admission_intensity, claim frequency, provider concentration"
    },
    {
        "module": "Pricing",
        "main_tables": "policies, claims, member_year_features, insured_members",
        "unit_of_analysis": "policy_id / member_id-policy-year",
        "key_variables": "premium_monthly, premium_annual, pricing_adequacy_ratio, deductible_amount, coverage_scope, expected annual cost, renewal_flag, cancellation_flag"
    },
    {
        "module": "Prospect Profiler",
        "main_tables": "prospect_survey, insured_members, policies",
        "unit_of_analysis": "prospect_id",
        "key_variables": "age, dependents_n, smoker_flag, bmi_group, chronic_condition_flag, price_sensitivity, coverage_preference, network_preference, preferred_plan_type"
    },
])

module_map

,module,main_tables,unit_of_analysis,key_variables
0,Risk Scoring,"insured_members, policies, claims, member_year...",member_id / member_id-year,"age, sex, chronic_condition_flag, chronic_cond..."
1,Fraud / Abuse,"claims, providers, provider_month_features, me...",claim_id / member_id / provider_id-month,"misuse_exposure_propensity, provider_archetype..."
2,Pricing,"policies, claims, member_year_features, insure...",policy_id / member_id-policy-year,"premium_monthly, premium_annual, pricing_adequ..."
3,Prospect Profiler,"prospect_survey, insured_members, policies",prospect_id,"age, dependents_n, smoker_flag, bmi_group, chr..."


In [8]:
## Target analytical tables
target_tables = pd.DataFrame([
    {
        "target_table": "member_master.csv",
        "description": "Member + policy unified base for portfolio analysis and first segmentation",
        "source_tables": "insured_members + policies",
        "primary_key": "member_id"
    },
    {
        "target_table": "claims_analytical_base.csv",
        "description": "Claims enriched with member, policy and provider information",
        "source_tables": "claims + insured_members + policies + providers",
        "primary_key": "claim_id"
    },
    {
        "target_table": "provider_master.csv",
        "description": "Provider consolidated analytical table for anomaly and cost analysis",
        "source_tables": "providers + provider_month_features + claims aggregates",
        "primary_key": "provider_id / provider_id-month"
    },
    {
        "target_table": "prospect_model_base.csv",
        "description": "Survey-based prospect table for plan recommendation",
        "source_tables": "prospect_survey + archetype mapping logic",
        "primary_key": "prospect_id"
    },
])

target_tables

,target_table,description,source_tables,primary_key
0,member_master.csv,Member + policy unified base for portfolio ana...,insured_members + policies,member_id
1,claims_analytical_base.csv,"Claims enriched with member, policy and provid...",claims + insured_members + policies + providers,claim_id
2,provider_master.csv,Provider consolidated analytical table for ano...,providers + provider_month_features + claims a...,provider_id / provider_id-month
3,prospect_model_base.csv,Survey-based prospect table for plan recommend...,prospect_survey + archetype mapping logic,prospect_id


In [9]:
## First-pass data dictionary
dictionary_rows = []

for table_name, df in {
    "insured_members": members,
    "policies": policies,
    "providers": providers,
    "claims": claims,
    "member_year_features": member_year,
    "provider_month_features": provider_month,
    "prospect_survey": prospects,
}.items():
    for col in df.columns:
        dictionary_rows.append({
            "table": table_name,
            "column": col,
            "dtype": str(df[col].dtype),
            "non_null_n": int(df[col].notna().sum()),
            "null_n": int(df[col].isna().sum()),
            "n_unique": int(df[col].nunique(dropna=True))
        })

data_dictionary = pd.DataFrame(dictionary_rows)
data_dictionary.head(20)

,table,column,dtype,non_null_n,null_n,n_unique
0,insured_members,member_id,object,4000,0,4000
1,insured_members,policy_id,object,4000,0,4000
2,insured_members,join_date,object,4000,0,1068
3,insured_members,age,int64,4000,0,63
4,insured_members,age_band,object,4000,0,6
5,insured_members,sex,object,4000,0,2
6,insured_members,region,object,4000,0,8
7,insured_members,city_tier,object,4000,0,3
8,insured_members,socioeconomic_band,object,4000,0,5
9,insured_members,employment_status,object,4000,0,5


In [10]:
# Optional: save draft data dictionary
output_path = PROJECT_ROOT / "docs" / "data_dictionary_draft.csv"
output_path.parent.mkdir(parents=True, exist_ok=True)
data_dictionary.to_csv(output_path, index=False)

print("Saved:", output_path)

Saved: C:\Users\dolivares\Desktop\novares-securehealth\docs\data_dictionary_draft.csv


In [11]:
## Analytical conclusions
conclusions = [
    "The project already has the minimum table set required to structure all four core modules.",
    "insured_members and policies can be merged immediately to create the first analytical base.",
    "claims and provider_month_features enable a strong future fraud / abuse module.",
    "pricing variables already exist and can support early pricing adequacy analysis.",
    "prospect_survey_synthetic allows future plan recommendation and archetype profiling."
]

for i, c in enumerate(conclusions, 1):
    print(f"{i}. {c}")

1. The project already has the minimum table set required to structure all four core modules.
2. insured_members and policies can be merged immediately to create the first analytical base.
3. claims and provider_month_features enable a strong future fraud / abuse module.
4. pricing variables already exist and can support early pricing adequacy analysis.
5. prospect_survey_synthetic allows future plan recommendation and archetype profiling.


## Next step
Build `member_master.csv` from:

- `insured_members.csv`
- `policies.csv`

in `03_member_master_build.ipynb`.